In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

# 1. Өгөгдлөө ачааллах
dataset = 'dataset.csv'
model_save_path = 'keypoint_classifier.hdf5'
tflite_save_path = 'keypoint_classifier.tflite'

# CSV файлыг унших (Header байхгүй гэж үзвэл)
df = pd.read_csv(dataset, header=None)

# X (42 координат) болон y (Label) салгах
X_dataset = df.iloc[:, 1:].values.astype('float32')
y_dataset = df.iloc[:, 0].values.astype('int32')

# Сургалтын болон шалгалтын өгөгдөл болгон хуваах (80% сургалт, 20% шалгалт)
X_train, X_test, y_train, y_test = train_test_split(X_dataset, y_dataset, train_size=0.8, random_state=42)

2026-03-29 17:02:29.063255: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-29 17:02:29.064669: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-29 17:02:29.092476: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-29 17:02:29.093173: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-29 17:02:29.715033: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Co

In [2]:
# Сургалтын болон шалгалтын өгөгдөл болгон хуваах (80% сургалт, 20% шалгалт)
X_train, X_test, y_train, y_test = train_test_split(X_dataset, y_dataset, train_size=0.8, random_state=42)

In [3]:
# 2. Модел байгуулах (Олон давхаргат нейрон сүлжээ)
NUM_CLASSES = len(np.unique(y_dataset)) # Нийт ангиллын тоо

model = tf.keras.models.Sequential([
    tf.keras.layers.Input((42, )), # 21 цэг * 2 (X,Y)
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2), # Хэт сурахаас (overfitting) сэргийлэх
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax') # Гаралт (Ангилал бүрийн магадлал)
])

model.summary() # Моделийн бүтцийг харах

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                2752      
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dropout_1 (Dropout)         (None, 32)                0         
                                                                 
 dense_2 (Dense)             (None, 16)                528       
                                                                 
 dense_3 (Dense)             (None, 10)                170       
                                                                 
Total params: 5530 (21.60 KB)
Trainable params: 5530 (21

2026-03-29 17:02:41.521283: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-29 17:02:41.548921: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [4]:
# 3. Моделыг сургах тохиргоо
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Сургах (100-200 удаа давталт буюу epochs хийхэд хангалттай)
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/100
195/195 [==============================] - 1s 1ms/step - loss: 1.6930 - accuracy: 0.4157 - val_loss: 1.0507 - val_accuracy: 0.7249
Epoch 2/100
195/195 [==============================] - 0s 1ms/step - loss: 0.8816 - accuracy: 0.7217 - val_loss: 0.4775 - val_accuracy: 0.8548
Epoch 3/100
195/195 [==============================] - 0s 990us/step - loss: 0.5335 - accuracy: 0.8284 - val_loss: 0.2962 - val_accuracy: 0.9332
Epoch 4/100
195/195 [==============================] - 0s 972us/step - loss: 0.3911 - accuracy: 0.8724 - val_loss: 0.2176 - val_accuracy: 0.9389
Epoch 5/100
195/195 [==============================] - 0s 1ms/step - loss: 0.3199 - accuracy: 0.8952 - val_loss: 0.1791 - val_accuracy: 0.9473
Epoch 6/100
195/195 [==============================] - 0s 1ms/step - loss: 0.2621 - accuracy: 0.9156 - val_loss: 0.1405 - val_accuracy: 0.9589
Epoch 7/100
195/195 [==============================] - 0s 1ms/step - loss: 0.2231 - accuracy: 0.9275 - val_loss: 0.1173 - val_accuracy: 0.

In [5]:
# 4. Моделыг TFLite формат руу хөрвүүлэх
# Эхлээд Keras моделиор хадгална
model.save(model_save_path)

# TFLite руу хөрвүүлэгч
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT] # Хэмжээг нь багасгаж оновчлох
tflite_quantized_model = converter.convert()

# TFLite файлыг хадгалах
with open(tflite_save_path, 'wb') as f:
    f.write(tflite_quantized_model)

print(f"Амжилттай! TFLite модел '{tflite_save_path}' нэрээр хадгалагдлаа.")

/home/enkush-3/miniconda3/envs/gesture_env/lib/python3.8/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


INFO:tensorflow:Assets written to: /tmp/tmpcj82fd5j/assets


INFO:tensorflow:Assets written to: /tmp/tmpcj82fd5j/assets


Амжилттай! TFLite модел 'keypoint_classifier.tflite' нэрээр хадгалагдлаа.


2026-03-29 17:03:37.375924: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
2026-03-29 17:03:37.375998: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-03-29 17:03:37.376305: I tensorflow/cc/saved_model/reader.cc:45] Reading SavedModel from: /tmp/tmpcj82fd5j
2026-03-29 17:03:37.377594: I tensorflow/cc/saved_model/reader.cc:91] Reading meta graph with tags { serve }
2026-03-29 17:03:37.377626: I tensorflow/cc/saved_model/reader.cc:132] Reading SavedModel debug info (if present) from: /tmp/tmpcj82fd5j
2026-03-29 17:03:37.381371: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:375] MLIR V1 optimization pass is not enabled
2026-03-29 17:03:37.382492: I tensorflow/cc/saved_model/loader.cc:231] Restoring SavedModel bundle.
2026-03-29 17:03:37.423739: I tensorflow/cc/saved_model/loader.cc:215] Running initialization op on SavedModel bundle at path: /tmp/tmpcj82fd5j
2026-03